<a href="https://colab.research.google.com/github/RabeeaAnjum/ML_INTERNSHIP_NEXTGEN_task2/blob/main/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%%writefile app2.py
import streamlit as st
import pickle
import numpy as np

# Load the saved models and scaler
model_rf = pickle.load(open('model_rf.pkl', 'rb'))  # Load RandomForest model
model_lr = pickle.load(open('model_lr.pkl', 'rb'))  # Load Logistic Regression model
model_gb = pickle.load(open('model_gb.pkl', 'rb'))  # Load Gradient Boosting model
scaler = pickle.load(open('scaler.pkl', 'rb'))      # Load the scaler

st.set_page_config(page_title="Customer Churn Prediction", layout="centered")

st.title("📊 Customer Churn Prediction System")
st.markdown("### Enter Customer Details")

# ----------- INPUT FIELDS -----------

# Numerical
tenure = st.slider("Tenure (Months)", 0, 72, 12)
monthly_charges = st.number_input("Monthly Charges", value=50.0)
total_charges = st.number_input("Total Charges", value=500.0)

# Gender
gender = st.selectbox("Gender", ["Male", "Female"])
gender = 1 if gender == "Male" else 0

# Senior Citizen
senior = st.selectbox("Senior Citizen", ["No", "Yes"])
senior = 1 if senior == "Yes" else 0

# Partner
partner = st.selectbox("Partner", ["No", "Yes"])
partner = 1 if partner == "Yes" else 0

# Dependents
dependents = st.selectbox("Dependents", ["No", "Yes"])
dependents = 1 if dependents == "Yes" else 0

# Phone Service
phone = st.selectbox("Phone Service", ["No", "Yes"])
phone = 1 if phone == "Yes" else 0

# Internet Service
internet = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
internet_map = {"DSL":0, "Fiber optic":1, "No":2}
internet = internet_map[internet]

# Online Security
security = st.selectbox("Online Security", ["No", "Yes"])
security = 1 if security == "Yes" else 0

# Tech Support
tech = st.selectbox("Tech Support", ["No", "Yes"])
tech = 1 if tech == "Yes" else 0

# Contract
contract = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
contract_map = {"Month-to-month":0, "One year":1, "Two year":2}
contract = contract_map[contract]

# Payment Method
payment = st.selectbox("Payment Method", [
    "Electronic check", "Mailed check",
    "Bank transfer", "Credit card"
])
payment_map = {
    "Electronic check":0,
    "Mailed check":1,
    "Bank transfer":2,
    "Credit card":3
}
payment = payment_map[payment]

# ----------- PREDICTION -----------

if st.button("🔍 Predict Churn"):

    input_data = np.array([[tenure, monthly_charges, total_charges,
                            gender, senior, partner, dependents,
                            phone, internet, security, tech,
                            contract, payment]])

    # Scale input data
    input_scaled = scaler.transform(input_data)

    # Predict using all models
    rf_pred = model_rf.predict(input_scaled)
    lr_pred = model_lr.predict(input_scaled)
    gb_pred = model_gb.predict(input_scaled)

    # Calculate probabilities for each model
    rf_prob = model_rf.predict_proba(input_scaled)[0][1]
    lr_prob = model_lr.predict_proba(input_scaled)[0][1]
    gb_prob = model_gb.predict_proba(input_scaled)[0][1]

    # Display results for all models
    st.write("### Model Predictions:")

    st.write(f"**Random Forest Prediction:** {'Churn' if rf_pred[0] == 1 else 'Stay'} (Probability: {rf_prob:.2f})")
    st.write(f"**Logistic Regression Prediction:** {'Churn' if lr_pred[0] == 1 else 'Stay'} (Probability: {lr_prob:.2f})")
    st.write(f"**Gradient Boosting Prediction:** {'Churn' if gb_pred[0] == 1 else 'Stay'} (Probability: {gb_prob:.2f})")

    # Choose the final prediction based on the best model or average of probabilities
    final_pred = 'Churn' if np.mean([rf_prob, lr_prob, gb_prob]) > 0.5 else 'Stay'
    st.write(f"### Final Prediction: {final_pred}")

Writing app2.py


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# --- 1. Load Data (Using a synthetic dataset for demonstration) ---
# In a real scenario, you'd load your actual customer churn dataset here.
# For example: df = pd.read_csv('your_churn_data.csv')

# Create a synthetic dataset for demonstration
data = {
    'tenure': np.random.randint(0, 72, 1000),
    'monthly_charges': np.random.uniform(20, 120, 1000),
    'total_charges': np.random.uniform(100, 5000, 1000),
    'gender': np.random.choice(['Male', 'Female'], 1000),
    'senior_citizen': np.random.choice(['No', 'Yes'], 1000),
    'partner': np.random.choice(['No', 'Yes'], 1000),
    'dependents': np.random.choice(['No', 'Yes'], 1000),
    'phone_service': np.random.choice(['No', 'Yes'], 1000),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], 1000),
    'online_security': np.random.choice(['No', 'Yes'], 1000),
    'tech_support': np.random.choice(['No', 'Yes'], 1000),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], 1000),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], 1000),
    'Churn': np.random.choice([0, 1], 1000, p=[0.75, 0.25]) # 25% churn rate
}
df = pd.DataFrame(data)

# Simulate some missing total_charges for new customers
df.loc[df['tenure'] == 0, 'total_charges'] = np.nan

print("Dataset loaded successfully:")
display(df.head())
display(df.info())

Dataset loaded successfully:


,tenure,monthly_charges,total_charges,gender,senior_citizen,partner,dependents,phone_service,internet_service,online_security,tech_support,contract,payment_method,Churn
0,7,113.063166,653.975275,Female,Yes,Yes,No,No,Fiber optic,Yes,No,One year,Electronic check,1
1,50,31.781126,1223.352638,Female,No,No,No,No,Fiber optic,Yes,No,Month-to-month,Electronic check,1
2,2,91.360678,1071.738536,Male,No,Yes,Yes,Yes,No,Yes,No,Two year,Mailed check,1
3,42,75.964612,823.196560,Male,Yes,Yes,Yes,Yes,No,Yes,No,One year,Mailed check,0
4,67,65.712635,1255.015278,Male,Yes,Yes,Yes,Yes,No,No,No,Two year,Mailed check,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tenure            1000 non-null   int64  
 1   monthly_charges   1000 non-null   float64
 2   total_charges     985 non-null    float64
 3   gender            1000 non-null   object 
 4   senior_citizen    1000 non-null   object 
 5   partner           1000 non-null   object 
 6   dependents        1000 non-null   object 
 7   phone_service     1000 non-null   object 
 8   internet_service  1000 non-null   object 
 9   online_security   1000 non-null   object 
 10  tech_support      1000 non-null   object 
 11  contract          1000 non-null   object 
 12  payment_method    1000 non-null   object 
 13  Churn             1000 non-null   int64  
dtypes: float64(2), int64(2), object(10)
memory usage: 109.5+ KB


None

Next, we'll preprocess the data by handling missing values, encoding categorical features, and scaling numerical features. This prepares the data for model training.

In [17]:
# --- 2. Data Preprocessing ---

# Handle missing values (e.g., in 'total_charges' for new customers)
# Replace NaN with 0 for new customers or using a suitable imputation strategy.
# For simplicity, we'll fill NaN with 0, assuming new customers have 0 total charges.
df['total_charges'] = df['total_charges'].fillna(0)

# Convert categorical features to numerical using one-hot encoding
# Identify categorical columns that are not already numerical and not the target 'Churn'
categorical_cols = df.select_dtypes(include='object').columns.tolist()
if 'gender' in categorical_cols: # Gender is binary, can be label encoded
    le = LabelEncoder()
    df['gender'] = le.fit_transform(df['gender'])
    categorical_cols.remove('gender')

# For other categorical columns (if any left), one-hot encode them
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Rename columns to be compatible with model training (remove special characters)
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

# Separate features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preprocessing complete. First 5 rows of scaled training data:")
display(pd.DataFrame(X_train_scaled, columns=X.columns).head())

Data preprocessing complete. First 5 rows of scaled training data:


,tenure,monthly_charges,total_charges,gender,senior_citizen_Yes,partner_Yes,dependents_Yes,phone_service_Yes,internet_service_Fiberoptic,internet_service_No,online_security_Yes,tech_support_Yes,contract_Oneyear,contract_Twoyear,payment_method_Creditcard,payment_method_Electroniccheck,payment_method_Mailedcheck
0,-0.235106,-0.967211,-0.251192,-0.948808,-0.944062,1.002503,1.067257,-1.005013,1.359035,-0.658553,0.968011,-0.946433,-0.678125,-0.699827,-0.561951,1.649524,-0.550392
1,0.098209,1.711305,0.547244,1.053953,-0.944062,-0.997503,1.067257,-1.005013,1.359035,-0.658553,0.968011,-0.946433,-0.678125,-0.699827,-0.561951,1.649524,-0.550392
2,-0.092257,1.118077,-0.130849,1.053953,1.059253,-0.997503,1.067257,-1.005013,-0.735816,-0.658553,0.968011,-0.946433,-0.678125,-0.699827,-0.561951,-0.606235,-0.550392
3,0.526757,-1.584728,-1.357523,1.053953,1.059253,1.002503,-0.936981,0.995012,1.359035,-0.658553,0.968011,1.056599,1.474654,-0.699827,-0.561951,-0.606235,1.816886
4,1.526702,0.575109,1.144575,1.053953,-0.944062,1.002503,-0.936981,-1.005013,1.359035,-0.658553,0.968011,1.056599,-0.678125,1.428924,1.779513,-0.606235,-0.550392


Now, we will define and train the RandomForest, Logistic Regression, and Gradient Boosting models. These trained models, along with the fitted `scaler`, will then be saved using `pickle`.

In [19]:
# --- 3. Define and Train Models ---

# Initialize and train RandomForest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Initialize and train Logistic Regression model
lr_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' for small datasets
lr_model.fit(X_train_scaled, y_train)

# Initialize and train Gradient Boosting model
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_train)

print("All models and scaler have been defined and trained.")
print("You can now run the cell to save them using pickle.")

All models and scaler have been defined and trained.
You can now run the cell to save them using pickle.


In [21]:
import pickle


# Save RandomForest model
pickle.dump(rf_model, open('model_rf.pkl', 'wb'))  # Save the RandomForest model

# Save LogisticRegression model
pickle.dump(lr_model, open('model_lr.pkl', 'wb'))  # Save the Logistic Regression model

# Save GradientBoostingClassifier model
pickle.dump(gb_model, open('model_gb.pkl', 'wb'))  # Save the Gradient Boosting model

# Save the scaler used for feature scaling
pickle.dump(scaler, open('scaler.pkl', 'wb'))      # Save the scaler

In [22]:
!pip install streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 4s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [26]:
!streamlit cache clear

In [ ]:
!streamlit run app2.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://new-sites-hope.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.28.183:8501

